In [1]:
'''
Purpose of the notebook:
This notebook transforms the music tracks into three different representations:

1)Waveform (raw audio)

2)Mel spectrogram using the STFT

3 )CQT (Constant-Q-Transformation) spectrogram

After editing the 3 represantions, the input shapes are aligned for fair comparising
These extracted features are then used in the following notebooks for model training and evaluation.
'''

'\nPurpose of the notebook:\nThis notebook transforms the music tracks into three different representations:\n\n1)Waveform (raw audio)\n\n2)Mel spectrogram using the STFT\n\n3 )CQT (Constant-Q-Transformation) spectrogram\n\nAfter editing the 3 represantions, the input shapes are aligned for fair comparising\nThese extracted features are then used in the following notebooks for model training and evaluation.\n'

In [2]:
!pip install librosa
!pip install openpyxl

import os
import librosa
import numpy as np
from multiprocessing import Pool
from tqdm import tqdm
import traceback
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 102.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 134.6 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 115.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [librosa]m7/8 [librosa]]


In [3]:
df = pd.read_csv("/work/nlpminifolder/extracted_files/fma_small_merged.csv")

df.head()


,filepath,filename,track_id,audio_path,top_genres,num_labels,label_Rock,label_Electronic,label_Pop,label_Folk,label_Instrumental,label_HipHop,label_International,label_Classical,label_Jazz,label_Country,label_Blues,label_SoulRnB,mel_path
0,fma_small/018/018032.mp3,018032.mp3,18032,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/work/NLP-mini-project/datasets/fma/fma_clean/...
1,fma_small/018/018037.mp3,018037.mp3,18037,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/work/NLP-mini-project/datasets/fma/fma_clean/...
2,fma_small/018/018144.mp3,018144.mp3,18144,/datasets/fma/fma_clean/fma_selected/018/01814...,[10],1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,/work/NLP-mini-project/datasets/fma/fma_clean/...
3,fma_small/018/018159.mp3,018159.mp3,18159,/datasets/fma/fma_clean/fma_selected/018/01815...,[17],1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,/work/NLP-mini-project/datasets/fma/fma_clean/...
4,fma_small/018/018038.mp3,018038.mp3,18038,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/work/NLP-mini-project/datasets/fma/fma_clean/...


In [4]:
df = df.drop(columns=["mel_path"])


In [5]:
def make_path(track_id):
    tid_str = f"{int(track_id):06d}"   # 000123
    folder = tid_str[:3]               # 000
    return f"music/fma_small/{folder}/{tid_str}.mp3"
df["path"] = df["track_id"].apply(make_path)
df.head()


,filepath,filename,track_id,audio_path,top_genres,num_labels,label_Rock,label_Electronic,label_Pop,label_Folk,label_Instrumental,label_HipHop,label_International,label_Classical,label_Jazz,label_Country,label_Blues,label_SoulRnB,path
0,fma_small/018/018032.mp3,018032.mp3,18032,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,music/fma_small/018/018032.mp3
1,fma_small/018/018037.mp3,018037.mp3,18037,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,music/fma_small/018/018037.mp3
2,fma_small/018/018144.mp3,018144.mp3,18144,/datasets/fma/fma_clean/fma_selected/018/01814...,[10],1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,music/fma_small/018/018144.mp3
3,fma_small/018/018159.mp3,018159.mp3,18159,/datasets/fma/fma_clean/fma_selected/018/01815...,[17],1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,music/fma_small/018/018159.mp3
4,fma_small/018/018038.mp3,018038.mp3,18038,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,music/fma_small/018/018038.mp3


In [6]:
#df = df.drop(columns=["filepath"])
df = df.drop(columns=["path"])
df.head()

,filepath,filename,track_id,audio_path,top_genres,num_labels,label_Rock,label_Electronic,label_Pop,label_Folk,label_Instrumental,label_HipHop,label_International,label_Classical,label_Jazz,label_Country,label_Blues,label_SoulRnB
0,fma_small/018/018032.mp3,018032.mp3,18032,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,fma_small/018/018037.mp3,018037.mp3,18037,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,fma_small/018/018144.mp3,018144.mp3,18144,/datasets/fma/fma_clean/fma_selected/018/01814...,[10],1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,fma_small/018/018159.mp3,018159.mp3,18159,/datasets/fma/fma_clean/fma_selected/018/01815...,[17],1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,fma_small/018/018038.mp3,018038.mp3,18038,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [7]:
#1) waveform: transforming the music to a waveform for the entire 30 seconds
'''
import librosa
import numpy as np

TARGET_SR = 44100
TARGET_LEN = TARGET_SR * 30   # 30 seconds

def load_and_fix_waveform(path):
    y, sr = librosa.load(path, sr=TARGET_SR, mono=True)

    if len(y) < TARGET_LEN:
        y = np.pad(y, (0, TARGET_LEN - len(y)), mode="constant")
    else:
        y = y[:TARGET_LEN]

    return y.astype("float32")
import os

waveform_paths = []

for tid in df["track_id"].astype(int):
    tid6 = f"{tid:06d}"
    sub = tid6[:3]
    audio_path = f"../extracted_files/fma_small/{sub}/{tid6}.mp3"

    save_path = f"../extracted_files/waveforms/{tid6}.npy"  #-->Safed place

    if os.path.exists(audio_path):
        y = load_and_fix_waveform(audio_path)
        np.save(save_path, y)
        waveform_paths.append(save_path)
'''


'\nimport librosa\nimport numpy as np\n\nTARGET_SR = 44100\nTARGET_LEN = TARGET_SR * 30   # 30 seconds\n\ndef load_and_fix_waveform(path):\n    y, sr = librosa.load(path, sr=TARGET_SR, mono=True)\n\n    if len(y) < TARGET_LEN:\n        y = np.pad(y, (0, TARGET_LEN - len(y)), mode="constant")\n    else:\n        y = y[:TARGET_LEN]\n\n    return y.astype("float32")\nimport os\n\nwaveform_paths = []\n\nfor tid in df["track_id"].astype(int):\n    tid6 = f"{tid:06d}"\n    sub = tid6[:3]\n    audio_path = f"../extracted_files/fma_small/{sub}/{tid6}.mp3"\n\n    save_path = f"../extracted_files/waveforms/{tid6}.npy"  #-->Safed place\n\n    if os.path.exists(audio_path):\n        y = load_and_fix_waveform(audio_path)\n        np.save(save_path, y)\n        waveform_paths.append(save_path)\n'

In [8]:
#adding the waveform to it

df["waveform_path"] = df["track_id"].apply(
    lambda x: f"/work/nlpminifolder/extracted_files/waveforms/{int(x):06d}.npy"
)


In [9]:
#checking the input shape of the waveform 

df["waveform_exists"] = df["waveform_path"].apply(os.path.exists)
bad_shapes = df[df["waveform_exists"]].copy()

wrong = []

for idx, row in bad_shapes.iterrows():
    y = np.load(row["waveform_path"])
    if y.shape[0] != 1323000:
        wrong.append((row["track_id"], y.shape))

wrong[:10], len(wrong)
#None waveforms with a wrong input shape

([], 0)

In [10]:
df["waveform_exists"].value_counts()
#2046 mp3 songs do not have a waveform

waveform_exists
True     8079
False    2046
Name: count, dtype: int64

In [11]:
df = df[df["waveform_path"].apply(os.path.exists)].copy()
#df = df.drop(columns=["waveform_exists"])
#df= df.drop(columns=["audio_path_generated"])
df.isnull().sum()

filepath               0
filename               0
track_id               0
audio_path             0
top_genres             0
num_labels             0
label_Rock             0
label_Electronic       0
label_Pop              0
label_Folk             0
label_Instrumental     0
label_HipHop           0
label_International    0
label_Classical        0
label_Jazz             0
label_Country          0
label_Blues            0
label_SoulRnB          0
waveform_path          0
waveform_exists        0
dtype: int64

In [12]:
#----------end waveform df-----------------------------------

In [13]:
#2)----------mel spectogram -----------------

In [14]:
#creating a mel spectogram with 30 seconds as input and 128 bins as frequency bins
'''
import os
import librosa
import numpy as np
from multiprocessing import Pool
from tqdm import tqdm
import traceback

# -------------------------------------------------------
# CONFIG — (now matching your real folder)
# -------------------------------------------------------
INPUT_ROOT = "extracted_files/fma_small"   # <- dein echter Ordner
OUTPUT_ROOT = "mels"                       
#ordner wo die gespeichert werden
SAMPLE_RATE = 32000
N_MELS = 128
WIN_LENGTH = int(0.025 * SAMPLE_RATE)  # 25ms
HOP_LENGTH = int(0.010 * SAMPLE_RATE)  # 10ms


# -------------------------------------------------------
# CREATE OUTPUT FOLDER STRUCTURE (000, 001, ..., 255)
# -------------------------------------------------------
for i in range(256):
    folder = os.path.join(OUTPUT_ROOT, f"{i:03d}")
    os.makedirs(folder, exist_ok=True)


def process_file(task):
    in_path, out_path = task

    if os.path.exists(out_path):
        return f"SKIPPED {out_path}"

    try:
        # Load audio
        y, sr = librosa.load(in_path, sr=SAMPLE_RATE, mono=True)

        # Mel spectrogram
        mel = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_fft=WIN_LENGTH,
            hop_length=HOP_LENGTH,
            win_length=WIN_LENGTH,
            n_mels=N_MELS,
            power=2.0
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        # Save
        np.save(out_path, mel_db)
        return f"OK {out_path}"

    except Exception as e:
        error_msg = f"ERROR {in_path}: {repr(e)}"
        with open(os.path.join(OUTPUT_ROOT, "errors.log"), "a") as f:
            f.write(error_msg + "\n")
            f.write(traceback.format_exc() + "\n")
        return error_msg


# -------------------------------------------------------
# BUILD TASK LIST
# -------------------------------------------------------
tasks = []

for root, dirs, files in os.walk(INPUT_ROOT):
    for f in files:
        if not f.endswith(".mp3"):
            continue

        # Beispiel: extracted_files/fma_small/093/093522.mp3
        track_id = f[:-4]          # 093522
        subfolder = track_id[:3]   # 093

        in_path = os.path.join(root, f)
        out_path = os.path.join(OUTPUT_ROOT, subfolder, track_id + ".npy")

        tasks.append((in_path, out_path))

print("Found", len(tasks), "audio files.")


# -------------------------------------------------------
# MULTIPROCESSING
# -------------------------------------------------------
if __name__ == "__main__":
    num_workers = max(os.cpu_count() - 1, 1)
    print("Using", num_workers, "workers...")

    with Pool(num_workers) as pool:
        for result in tqdm(pool.imap_unordered(process_file, tasks),
                           total=len(tasks), ncols=100):
            pass

    print("Done! All mels saved to:", OUTPUT_ROOT)
'''

'\nimport os\nimport librosa\nimport numpy as np\nfrom multiprocessing import Pool\nfrom tqdm import tqdm\nimport traceback\n\n# -------------------------------------------------------\n# CONFIG — (now matching your real folder)\n# -------------------------------------------------------\nINPUT_ROOT = "extracted_files/fma_small"   # <- dein echter Ordner\nOUTPUT_ROOT = "mels"                       \n#ordner wo die gespeichert werden\nSAMPLE_RATE = 32000\nN_MELS = 128\nWIN_LENGTH = int(0.025 * SAMPLE_RATE)  # 25ms\nHOP_LENGTH = int(0.010 * SAMPLE_RATE)  # 10ms\n\n\n# -------------------------------------------------------\n# CREATE OUTPUT FOLDER STRUCTURE (000, 001, ..., 255)\n# -------------------------------------------------------\nfor i in range(256):\n    folder = os.path.join(OUTPUT_ROOT, f"{i:03d}")\n    os.makedirs(folder, exist_ok=True)\n\n\ndef process_file(task):\n    in_path, out_path = task\n\n    if os.path.exists(out_path):\n        return f"SKIPPED {out_path}"\n\n    try:

In [15]:
def make_mel_path(track_id):
    tid = f"{int(track_id):06d}"  # z.B. 018032
    folder = tid[:3]              # 018
    return f"/work/nlpminifolder/extracted_files/mels/{folder}/{tid}.npy"

df["mel_path"] = df["track_id"].apply(make_mel_path)


In [16]:
#df = df.drop(columns=["mel_exists"])
df.head()

,filepath,filename,track_id,audio_path,top_genres,num_labels,label_Rock,label_Electronic,label_Pop,label_Folk,...,label_HipHop,label_International,label_Classical,label_Jazz,label_Country,label_Blues,label_SoulRnB,waveform_path,waveform_exists,mel_path
0,fma_small/018/018032.mp3,018032.mp3,18032,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/work/nlpminifolder/extracted_files/waveforms/...,True,/work/nlpminifolder/extracted_files/mels/018/0...
1,fma_small/018/018037.mp3,018037.mp3,18037,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/work/nlpminifolder/extracted_files/waveforms/...,True,/work/nlpminifolder/extracted_files/mels/018/0...
2,fma_small/018/018144.mp3,018144.mp3,18144,/datasets/fma/fma_clean/fma_selected/018/01814...,[10],1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,/work/nlpminifolder/extracted_files/waveforms/...,True,/work/nlpminifolder/extracted_files/mels/018/0...
3,fma_small/018/018159.mp3,018159.mp3,18159,/datasets/fma/fma_clean/fma_selected/018/01815...,[17],1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,/work/nlpminifolder/extracted_files/waveforms/...,True,/work/nlpminifolder/extracted_files/mels/018/0...
4,fma_small/018/018038.mp3,018038.mp3,18038,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,/work/nlpminifolder/extracted_files/waveforms/...,True,/work/nlpminifolder/extracted_files/mels/018/0...


In [17]:
#checking for non existing rows for mel_path
import os

df["mel_exists"] = df["mel_path"].apply(os.path.exists)
df["mel_exists"].value_counts()
#none

mel_exists
True    8079
Name: count, dtype: int64

In [18]:
#-----------------------------mel spectogram finished

In [19]:
#cqt spectogram

In [20]:

'''
# -------------------------------------------------------
# CONFIG — CQT mit 128 Bins (für fairen Vergleich mit Mel)
# -------------------------------------------------------
INPUT_ROOT = "/work/nlpminifolder/extracted_files/fma_small"
OUTPUT_ROOT = "/work/nlpminifolder/extracted_files/cqt"

SAMPLE_RATE = 32000
HOP_LENGTH = int(0.010 * SAMPLE_RATE)

N_BINS = 128                # identischer Frequenz-Shape wie Mel
BINS_PER_OCTAVE = 16        # sinnvoll für 128 Bins
FMIN = 32.7

# Output-Folder erstellen
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# -------------------------------------------------------
# MP3-Dateien sammeln
# -------------------------------------------------------
mp3s = []
for root, dirs, files in os.walk(INPUT_ROOT):
    for f in files:
        if f.endswith(".mp3"):
            full_path = os.path.join(root, f)
            if os.path.isfile(full_path) and os.path.getsize(full_path) > 5000:
                mp3s.append(full_path)

print("Gefundene gültige MP3s:", len(mp3s))

# -------------------------------------------------------
# CQT-Berechnung + Speichern
# -------------------------------------------------------
for mp3 in tqdm(mp3s):
    track_id = os.path.splitext(os.path.basename(mp3))[0]
    out_path = os.path.join(OUTPUT_ROOT, track_id + ".npy")

    # Skip, falls bereits vorhanden
    if os.path.exists(out_path):
        continue

    try:
        # AUDIO LADEN
        y, sr = librosa.load(mp3, sr=SAMPLE_RATE, mono=True)

        # CQT berechnen
        cqt = librosa.cqt(
            y,
            sr=sr,
            hop_length=HOP_LENGTH,
            n_bins=N_BINS,
            bins_per_octave=BINS_PER_OCTAVE,
            fmin=FMIN
        )

        cqt_mag = np.abs(cqt)
        cqt_db = librosa.amplitude_to_db(cqt_mag, ref=np.max)

        # SPEICHERN
        np.save(out_path, cqt_db)

    except Exception as e:
        # Fehler loggen und weitermachen
        with open(os.path.join(OUTPUT_ROOT, "cqt_errors.log"), "a") as f:
            f.write(f"{mp3}: {repr(e)}\n")
            f.write(traceback.format_exc() + "\n\n")
        continue

print("Fertig! CQT gespeichert in:", OUTPUT_ROOT)
'''

'\n# -------------------------------------------------------\n# CONFIG — CQT mit 128 Bins (für fairen Vergleich mit Mel)\n# -------------------------------------------------------\nINPUT_ROOT = "/work/nlpminifolder/extracted_files/fma_small"\nOUTPUT_ROOT = "/work/nlpminifolder/extracted_files/cqt"\n\nSAMPLE_RATE = 32000\nHOP_LENGTH = int(0.010 * SAMPLE_RATE)\n\nN_BINS = 128                # identischer Frequenz-Shape wie Mel\nBINS_PER_OCTAVE = 16        # sinnvoll für 128 Bins\nFMIN = 32.7\n\n# Output-Folder erstellen\nos.makedirs(OUTPUT_ROOT, exist_ok=True)\n\n# -------------------------------------------------------\n# MP3-Dateien sammeln\n# -------------------------------------------------------\nmp3s = []\nfor root, dirs, files in os.walk(INPUT_ROOT):\n    for f in files:\n        if f.endswith(".mp3"):\n            full_path = os.path.join(root, f)\n            if os.path.isfile(full_path) and os.path.getsize(full_path) > 5000:\n                mp3s.append(full_path)\n\nprint("Gef

In [21]:
#adding cqt spectograms to df
def make_cqt_path(track_id):
    tid = f"{int(track_id):06d}"
    return f"/work/nlpminifolder/extracted_files/cqt/{tid}.npy"
df["cqt_path"] = df["track_id"].apply(make_cqt_path)
df.head()



,filepath,filename,track_id,audio_path,top_genres,num_labels,label_Rock,label_Electronic,label_Pop,label_Folk,...,label_Classical,label_Jazz,label_Country,label_Blues,label_SoulRnB,waveform_path,waveform_exists,mel_path,mel_exists,cqt_path
0,fma_small/018/018032.mp3,018032.mp3,18032,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,/work/nlpminifolder/extracted_files/waveforms/...,True,/work/nlpminifolder/extracted_files/mels/018/0...,True,/work/nlpminifolder/extracted_files/cqt/018032...
1,fma_small/018/018037.mp3,018037.mp3,18037,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,/work/nlpminifolder/extracted_files/waveforms/...,True,/work/nlpminifolder/extracted_files/mels/018/0...,True,/work/nlpminifolder/extracted_files/cqt/018037...
2,fma_small/018/018144.mp3,018144.mp3,18144,/datasets/fma/fma_clean/fma_selected/018/01814...,[10],1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,/work/nlpminifolder/extracted_files/waveforms/...,True,/work/nlpminifolder/extracted_files/mels/018/0...,True,/work/nlpminifolder/extracted_files/cqt/018144...
3,fma_small/018/018159.mp3,018159.mp3,18159,/datasets/fma/fma_clean/fma_selected/018/01815...,[17],1.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,/work/nlpminifolder/extracted_files/waveforms/...,True,/work/nlpminifolder/extracted_files/mels/018/0...,True,/work/nlpminifolder/extracted_files/cqt/018159...
4,fma_small/018/018038.mp3,018038.mp3,18038,/datasets/fma/fma_clean/fma_selected/018/01803...,[2],1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,/work/nlpminifolder/extracted_files/waveforms/...,True,/work/nlpminifolder/extracted_files/mels/018/0...,True,/work/nlpminifolder/extracted_files/cqt/018038...


In [22]:
# checking for non existing rows for cqt_path
df["cqt_exists"] = df["cqt_path"].apply(os.path.exists)
df["cqt_exists"].value_counts()
#no NUlls

cqt_exists
True    8079
Name: count, dtype: int64

In [23]:
#transforming into cqt spectograms finished

In [24]:
#---------------adjusting input shape 

In [25]:
'''
Goal: Ensure a fair comparison between all audio representations.

Problem:
Different input lengths (time axis or waveform duration) would make the models
incomparable, because they would not receive the same amount of temporal
information.

Solution:
All representations are normalized to the same 30-second duration.
The waveform is fixed to 44,100 Hz × 30 s = 1,323,000 samples, while Mel and
CQT spectrograms are padded/trimmed to 3000 time frames (≈ 30 s with a 10 ms hop).
Thus, all inputs encode the same temporal content.

Note:
For the multi-channel model (Mel + CQT), both spectrograms must also share the
exact same input shape, so aligning Mel and CQT to (128, 3000) is required.
'''


'\nGoal: Ensure a fair comparison between all audio representations.\n\nProblem:\nDifferent input lengths (time axis or waveform duration) would make the models\nincomparable, because they would not receive the same amount of temporal\ninformation.\n\nSolution:\nAll representations are normalized to the same 30-second duration.\nThe waveform is fixed to 44,100 Hz × 30 s = 1,323,000 samples, while Mel and\nCQT spectrograms are padded/trimmed to 3000 time frames (≈ 30 s with a 10 ms hop).\nThus, all inputs encode the same temporal content.\n\nNote:\nFor the multi-channel model (Mel + CQT), both spectrograms must also share the\nexact same input shape, so aligning Mel and CQT to (128, 3000) is required.\n'

In [26]:
#padding trimming for both mel and cqt, this ensures all representations have the same length within mel and cqt spectogram 
def pad_or_trim_2d(arr, target_time=3000):
    """
    Ensures arr has shape (freq_bins, target_time).
    Pads or trims along the time axis.
    """
    freq_bins, time_bins = arr.shape
    
    if time_bins < target_time:
        pad_width = target_time - time_bins
        arr = np.pad(arr, ((0, 0), (0, pad_width)), mode="constant")
    elif time_bins > target_time:
        arr = arr[:, :target_time]

    return arr.astype("float32")


In [27]:
#padding for waveform ( seperate since 1d != 2d cnn)
def pad_or_trim_1d(arr, target_len):
    """
    Ensures waveform has length target_len.
    """
    L = len(arr)

    if L < target_len:
        arr = np.pad(arr, (0, target_len - L), mode="constant")
    elif L > target_len:
        arr = arr[:target_len]

    return arr.astype("float32")


In [28]:
def process_folder(input_folder, pad_func, target):
    for root, dirs, files in os.walk(input_folder):
        for f in tqdm(files, desc=f"Processing {input_folder}"):
            if f.endswith(".npy"):
                full = os.path.join(root, f)
                arr = np.load(full)
                arr = pad_func(arr, target)
                np.save(full, arr)


In [29]:
# Apply consistent padding/trimming so that mel, CQT, and waveform inputs all share a fixed, comparable shape.
process_folder("/work/nlpminifolder/extracted_files/mels", pad_or_trim_2d, 3000)
process_folder("/work/nlpminifolder/extracted_files/cqt", pad_or_trim_2d, 3000)
TARGET_LEN = 44100 * 30
process_folder("/work/nlpminifolder/extracted_files/waveforms", pad_or_trim_1d, TARGET_LEN)


Processing /work/nlpminifolder/extracted_files/mels: 100%|██████████| 1/1 [00:00<00:00, 16448.25it/s]
Processing /work/nlpminifolder/extracted_files/mels: 100%|██████████| 68/68 [00:00<00:00, 162.03it/s]
Processing /work/nlpminifolder/extracted_files/mels: 0it [00:00, ?it/s]
Processing /work/nlpminifolder/extracted_files/mels: 100%|██████████| 9/9 [00:00<00:00, 221.06it/s]
Processing /work/nlpminifolder/extracted_files/mels: 0it [00:00, ?it/s]
Processing /work/nlpminifolder/extracted_files/mels: 100%|██████████| 51/51 [00:00<00:00, 130.75it/s]
Processing /work/nlpminifolder/extracted_files/mels: 0it [00:00, ?it/s]
Processing /work/nlpminifolder/extracted_files/mels: 100%|██████████| 31/31 [00:00<00:00, 267.29it/s]
Processing /work/nlpminifolder/extracted_files/mels: 100%|██████████| 62/62 [00:00<00:00, 187.88it/s]
Processing /work/nlpminifolder/extracted_files/mels: 100%|██████████| 4/4 [00:00<00:00, 91.60it/s]
Processing /work/nlpminifolder/extracted_files/mels: 0it [00:00, ?it/s]
Pro

In [30]:
#checking the input shapes if it worked
def check_shapes(folder):
    shapes = set()
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.endswith(".npy"):
                arr = np.load(os.path.join(root, f))
                shapes.add(arr.shape)
    return shapes

print("MEL  :", check_shapes("/work/nlpminifolder/extracted_files/mels"))
print("CQT  :", check_shapes("/work/nlpminifolder/extracted_files/cqt"))
print("WAVE :", check_shapes("/work/nlpminifolder/extracted_files/waveforms"))


MEL  : {(128, 3000)}
CQT  : {(128, 3000)}
WAVE : {(1323000,)}


In [31]:
#-----------input chanel alignment finished

In [32]:
#------creating df_audio for actual comparisment of representation performance
'''

# 1) Copy dataframe
df_audio = df

# 2) Drop unnecessary existence-check columns
cols_to_drop = ["mel_exists", "waveform_exists", "cqt_exists"]

df_audio = df_audio.drop(columns=cols_to_drop, errors="ignore")

# 3) Save to Excel
output_path = "/work/nlpminifolder/extracted_files/df_audio.xlsx"
df_audio.to_excel(output_path, index=False)

print("Excel saved to:", output_path)
'''

Excel saved to: /work/nlpminifolder/extracted_files/df_audio.xlsx
